In [ ]:
# ✅ Create the dataset split files properly
import os

os.makedirs("/content/WildDeepfake/videos", exist_ok=True)

# Create train.txt and val.txt (as plain text files)
with open("/content/train.txt", "w") as f:
    f.write("""/content/WildDeepfake/videos/fake_video_001 1
/content/WildDeepfake/videos/fake_video_002 1
/content/WildDeepfake/videos/real_video_001 0
/content/WildDeepfake/videos/real_video_002 0
""")

with open("/content/val.txt", "w") as f:
    f.write("""/content/WildDeepfake/videos/fake_video_003 1
/content/WildDeepfake/videos/real_video_003 0
""")

print("✅ train.txt and val.txt created successfully!")


✅ train.txt and val.txt created successfully!


In [ ]:
import os
import cv2
import torch
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# ======================
# Configuration
# ======================
class cfg:
    data_root = "/content/WildDeepfake/videos"
    num_epochs = 2
    batch_size = 2
    lr = 1e-4
    num_frames = 16
    img_size = 224
    device = "cuda" if torch.cuda.is_available() else "cpu"

# ======================
# Dataset
# ======================
class VideoDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self._load_samples()

    def _load_samples(self):
        for label_dir in os.listdir(self.root_dir):
            label_path = os.path.join(self.root_dir, label_dir)
            if not os.path.isdir(label_path):
                continue
            label = 1 if "fake" in label_dir.lower() else 0
            for fname in os.listdir(label_path):
                if fname.endswith((".mp4", ".avi", ".mov")):
                    self.samples.append((os.path.join(label_path, fname), label))
        print(f"Loaded {len(self.samples)} videos from {self.root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = self._read_video(video_path)

        # Randomly sample frames if too long
        if len(frames) >= cfg.num_frames:
            indices = sorted(random.sample(range(len(frames)), cfg.num_frames))
            frames = [frames[i] for i in indices]
        else:
            # pad if too short
            while len(frames) < cfg.num_frames:
                frames.append(frames[-1])

        # Apply transform
        if self.transform:
            frames = [self.transform(frame) for frame in frames]

        frames = torch.stack(frames)
        return frames, torch.tensor(label, dtype=torch.long)

    def _read_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # Force 3-channel RGB and resize
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (cfg.img_size, cfg.img_size))
            frames.append(frame)
        cap.release()

        # Handle empty videos
        if len(frames) == 0:
            frames = [np.zeros((cfg.img_size, cfg.img_size, 3), dtype=np.uint8)]

        return frames

# ======================
# Model Definition
# ======================
class VideoClassifier(nn.Module):
    def __init__(self):
        super(VideoClassifier, self).__init__()
        base_model = models.resnet50(weights="IMAGENET1K_V1")
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
        self.fc = nn.Linear(2048, 2)

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        features = self.feature_extractor(x)
        features = features.view(B, T, -1)
        features = features.mean(dim=1)  # temporal average
        out = self.fc(features)
        return out

# ======================
# Training Utils
# ======================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc = 0, 0
    for X, y in tqdm(loader):
        X, y = X.to(cfg.device), y.to(cfg.device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += (outputs.argmax(1) == y).sum().item()
    return total_loss / len(loader), total_acc / len(loader.dataset)

def validate(model, loader, criterion):
    model.eval()
    total_loss, total_acc = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(cfg.device), y.to(cfg.device)
            outputs = model(X)
            loss = criterion(outputs, y)
            total_loss += loss.item()
            total_acc += (outputs.argmax(1) == y).sum().item()
    return total_loss / len(loader), total_acc / len(loader.dataset)

# ======================
# Main Training Script
# ======================
if __name__ == "__main__":
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
    ])

    dataset = VideoDataset(cfg.data_root, transform)
    train_loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2)

    model = VideoClassifier().to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = nn.CrossEntropyLoss()

    print("Starting training...")
    for epoch in range(cfg.num_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        print(f"Epoch {epoch+1}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.4f}")

    torch.save(model.state_dict(), "stil_best_model.pth")
    print("✅ Training complete. Model saved as stil_best_model.pth")


Loaded 0 videos from /content/WildDeepfake/videos


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
!python /content/STIL_best_model_train.py


  0% 0/2 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/content/STIL_best_model_train.py", line 141, in <module>
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/STIL_best_model_train.py", line 97, in train_one_epoch
    for X, y in tqdm(loader):
                ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 734, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1516, in _next_data
    return self._process_data(data, worker_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line

In [ ]:
# ==========================================
# SIMPLE STIL-STYLE DEEPFAKE DETECTOR (Colab Ready)
# ==========================================
!pip install torch torchvision tqdm --quiet

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
import numpy as np
import random

# ------------------------------
# 1️⃣ Simulated Dataset (fake vs real)
# ------------------------------
class SyntheticDeepfakeDataset(Dataset):
    def __init__(self, n_samples=200, img_size=224, clip_len=4):
        self.n_samples = n_samples
        self.img_size = img_size
        self.clip_len = clip_len
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size, img_size))
        ])

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        label = random.randint(0, 1)  # 0 = real, 1 = fake
        frames = []
        for _ in range(self.clip_len):
            if label == 0:
                frame = np.ones((self.img_size, self.img_size, 3), np.float32) * random.uniform(0.8, 1.0)
            else:
                frame = np.random.rand(self.img_size, self.img_size, 3).astype(np.float32) * random.uniform(0.5, 1.0)
            frames.append(torch.tensor(frame).permute(2, 0, 1))
        clip = torch.stack(frames)  # shape: [clip_len, 3, H, W]
        return clip, torch.tensor(label, dtype=torch.long)

# ------------------------------
# 2️⃣ Simple STIL-like Model (ResNet + Temporal Conv)
# ------------------------------
class SimpleSTIL(nn.Module):
    def __init__(self, num_classes=2, clip_len=4):
        super(SimpleSTIL, self).__init__()
        self.clip_len = clip_len
        base = models.resnet18(weights='IMAGENET1K_V1')
        self.feature_extractor = nn.Sequential(*list(base.children())[:-1])
        self.temporal_conv = nn.Conv1d(512, 128, kernel_size=3, padding=1)
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        feats = self.feature_extractor(x).view(B, T, 512)
        feats = feats.permute(0, 2, 1)
        temporal = torch.mean(self.temporal_conv(feats), dim=2)
        out = self.classifier(temporal)
        return out

# ------------------------------
# 3️⃣ Training Utilities
# ------------------------------
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc = 0, 0
    for clips, labels in tqdm(loader, desc="Training"):
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), total_acc / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc = 0, 0
    with torch.no_grad():
        for clips, labels in loader:
            clips, labels = clips.to(device), labels.to(device)
            outputs = model(clips)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            total_acc += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), total_acc / len(loader.dataset)

# ------------------------------
# 4️⃣ Main Script
# ------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
train_data = SyntheticDeepfakeDataset(n_samples=200)
val_data = SyntheticDeepfakeDataset(n_samples=50)
train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)

model = SimpleSTIL().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(3):
    tr_loss, tr_acc = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train Acc={tr_acc:.3f}, Val Acc={val_acc:.3f}")

print("\n✅ Training complete — Model ran successfully with synthetic data!")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 146MB/s]
Training: 100%|██████████| 50/50 [03:14<00:00,  3.89s/it]


Epoch 1: Train Acc=0.925, Val Acc=1.000


Training: 100%|██████████| 50/50 [03:11<00:00,  3.83s/it]


Epoch 2: Train Acc=0.955, Val Acc=1.000


Training: 100%|██████████| 50/50 [03:12<00:00,  3.86s/it]


Epoch 3: Train Acc=0.940, Val Acc=1.000

✅ Training complete — Model ran successfully with synthetic data!


In [ ]:
# creates: /content/dataset/real/clip0/*.jpg and /content/dataset/fake/clip0/*.jpg etc
import os
from PIL import Image
import numpy as np

DATA_ROOT = "/content/dataset"
CLASSES = ["real", "fake"]
CLIP_PER_CLASS = 3
FRAMES_PER_CLIP = 8
IMG_SIZE = 224

os.makedirs(DATA_ROOT, exist_ok=True)
for cls in CLASSES:
    cls_root = os.path.join(DATA_ROOT, cls)
    os.makedirs(cls_root, exist_ok=True)
    for clip_i in range(CLIP_PER_CLASS):
        clip_folder = os.path.join(cls_root, f"clip{clip_i:03d}")
        os.makedirs(clip_folder, exist_ok=True)
        for f_i in range(FRAMES_PER_CLIP):
            # simple synthetic content: real = bright gray, fake = random noise
            if cls == "real":
                arr = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * int(200 + 20 * (f_i % 2))
            else:
                arr = (np.random.rand(IMG_SIZE, IMG_SIZE, 3) * 255).astype(np.uint8)
            img = Image.fromarray(arr)
            img.save(os.path.join(clip_folder, f"frame{f_i:03d}.jpg"))

print("Created synthetic dataset at:", DATA_ROOT)
# List a bit:
for cls in CLASSES:
    print(cls, "->", os.listdir(os.path.join(DATA_ROOT, cls))[:5])


Created synthetic dataset at: /content/dataset
real -> ['clip000', 'clip001', 'clip002']
fake -> ['clip000', 'clip001', 'clip002']


In [ ]:
# Colab-ready full training script for a simple STIL-style deepfake detector.
# Supports datasets organized as video files in class subfolders (recommended),
# OR per-clip image folders. Uses decord (preferred) with torchvision fallback.

!pip install torch torchvision tqdm decord --quiet

import os
import glob
import random
from pathlib import Path
from typing import List, Optional

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.io import read_video  # fallback if decord missing
from PIL import Image

# ------------------------------
# Config (change as needed)
# ------------------------------
DATA_ROOT = "/content/dataset"   # set to your mounted drive or dataset path
# expected structure:
# DATA_ROOT/
#    real/
#      video1.mp4
#      video2.mp4
#    fake/
#      videoA.mp4
#      videoB.mp4
CLASSES = ["real", "fake"]
CLIP_LEN = 4
IMG_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 2
LR = 1e-4
EPOCHS = 5
CHECKPOINT_DIR = "/content/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------
# Video reader helper (decord preferred)
# ------------------------------
try:
    from decord import VideoReader, cpu as decord_cpu
    DECORD_AVAILABLE = True
except Exception:
    DECORD_AVAILABLE = False

def read_video_frames(path: str, num_frames: int) -> np.ndarray:
    """
    Return frames as numpy array shape (T, H, W, C), float32 in [0,1].
    Tries decord first, falls back to torchvision.read_video.
    """
    if DECORD_AVAILABLE:
        vr = VideoReader(path, ctx=decord_cpu(0))
        total = len(vr)
        if total == 0:
            raise RuntimeError(f"Empty video: {path}")
        # sample evenly
        idx = np.linspace(0, total - 1, num_frames).astype(int)
        frames = vr.get_batch(idx).asnumpy()  # (T, H, W, C), uint8
        frames = frames.astype(np.float32) / 255.0
        return frames
    else:
        # torchvision fallback (may be slower)
        vid, _, info = read_video(path, pts_unit='sec')  # vid shape (T, H, W, C), uint8
        if vid.numel() == 0:
            raise RuntimeError(f"Empty video (torchvision): {path}")
        vid = vid.numpy().astype(np.float32) / 255.0
        total = len(vid)
        idx = np.linspace(0, total - 1, num_frames).astype(int)
        frames = vid[idx]
        return frames

# ------------------------------
# Dataset: video files OR folder-of-frames
# ------------------------------
class DeepfakeVideoDataset(Dataset):
    """
    Assumes DATA_ROOT/<class_name> contains video files (.mp4/.avi/etc) OR subfolders with images (per clip).
    Returns: clip tensor [T, C, H, W], label (0..num_classes-1)
    """
    def __init__(self, root: str, classes: List[str], clip_len: int = 4, img_size: int = 224,
                 transform: Optional[transforms.Compose] = None, max_files_per_class: Optional[int] = None):
        self.root = Path(root)
        self.classes = classes
        self.clip_len = clip_len
        self.img_size = img_size
        self.transform = transform
        self.samples = []  # list of (path, label, mode) where mode in {"video","frames_dir"}
        exts = (".mp4", ".avi", ".mov", ".mkv", ".webm")
        for i, cls in enumerate(classes):
            cls_path = self.root / cls
            if not cls_path.exists():
                continue
            # 1) video files
            vids = [str(p) for ext in exts for p in cls_path.glob(f"*{ext}")]
            # also accept any file-like with ext
            vids += [str(p) for p in cls_path.iterdir() if p.suffix.lower() in exts]
            # 2) frame subfolders (each subfolder is a clip)
            frame_dirs = [str(p) for p in cls_path.iterdir() if p.is_dir()]
            # collect
            for v in vids:
                self.samples.append((v, i, "video"))
            for d in frame_dirs:
                # check if contains images
                img_files = list(Path(d).glob("*.jpg")) + list(Path(d).glob("*.png"))
                if len(img_files) > 0:
                    self.samples.append((d, i, "frames_dir"))
            if max_files_per_class:
                # limit per-class if requested
                self.samples = self.samples[:max_files_per_class * (i+1)]
        if len(self.samples) == 0:
            raise RuntimeError(f"No samples found in {root}. Check structure.")
        # shuffle samples
        random.shuffle(self.samples)

        # default transforms if not provided
        if transform is None:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
            ])

    def __len__(self):
        return len(self.samples)

    def _load_frames_from_dir(self, folder: str) -> np.ndarray:
        # load sorted image files
        paths = sorted(list(Path(folder).glob("*.jpg")) + list(Path(folder).glob("*.png")))
        if len(paths) == 0:
            raise RuntimeError(f"No images in folder: {folder}")
        total = len(paths)
        idx = np.linspace(0, total - 1, self.clip_len).astype(int)
        frames = []
        for i in idx:
            img = Image.open(paths[i]).convert("RGB")
            arr = np.asarray(img).astype(np.float32) / 255.0
            frames.append(arr)
        frames = np.stack(frames, axis=0)  # (T,H,W,C)
        return frames

    def __getitem__(self, idx):
        path, label, mode = self.samples[idx]
        if mode == "video":
            frames = read_video_frames(path, self.clip_len)  # (T,H,W,C)
        else:
            frames = self._load_frames_from_dir(path)  # (T,H,W,C)
        # apply transform to each frame separately and stack
        tensor_frames = []
        for f in frames:
            if self.transform:
                # transform expects HWC numpy or PIL; using ToPILImage in chain is ok
                t = self.transform((f * 255).astype(np.uint8))
            else:
                t = transforms.ToTensor()(f)
            tensor_frames.append(t)
        clip = torch.stack(tensor_frames)  # (T, C, H, W)
        return clip, torch.tensor(label, dtype=torch.long)

# ------------------------------
# Model
# ------------------------------
class SimpleSTIL(nn.Module):
    def __init__(self, num_classes=2, clip_len=4, pretrained=True):
        super(SimpleSTIL, self).__init__()
        self.clip_len = clip_len
        # using resnet18 pretrained
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        # drop final fc and pooling output remains (512,)
        self.feature_extractor = nn.Sequential(*list(base.children())[:-1])  # produces [B*T, 512, 1,1]
        # temporal conv: input channels = 512 features, applied over time
        self.temporal_conv = nn.Conv1d(512, 128, kernel_size=3, padding=1)
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x: [B, T, C, H, W]
        B, T, C, H, W = x.shape
        x = x.view(B * T, C, H, W)
        feats = self.feature_extractor(x)               # [B*T, 512, 1, 1]
        feats = feats.view(B, T, 512)                   # [B, T, 512]
        feats = feats.permute(0, 2, 1)                  # [B, 512, T]
        temporal = self.temporal_conv(feats)            # [B, 128, T]
        pooled = torch.mean(temporal, dim=2)           # [B, 128]
        out = self.classifier(pooled)                   # [B, num_classes]
        return out

# ------------------------------
# Training / Eval utilities
# ------------------------------
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    samples = 0
    for clips, labels in tqdm(loader, desc="Train"):
        clips, labels = clips.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(clips)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * clips.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        samples += clips.size(0)
    return total_loss / samples, correct / samples

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    samples = 0
    with torch.no_grad():
        for clips, labels in tqdm(loader, desc="Val"):
            clips, labels = clips.to(device), labels.to(device)
            outputs = model(clips)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * clips.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            samples += clips.size(0)
    return total_loss / samples, correct / samples

# ------------------------------
# Prepare data loaders
# ------------------------------
# create transforms for validation: deterministic
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_dataset = DeepfakeVideoDataset(DATA_ROOT, CLASSES, clip_len=CLIP_LEN, img_size=IMG_SIZE, transform=None)
val_dataset = None
# if you have separate train/val folders, change DATA_ROOT or supply separate roots.
# For demo: split train_dataset
val_split = 0.2
n_val = int(len(train_dataset) * val_split)
n_train = len(train_dataset) - n_val
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [n_train, n_val])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ------------------------------
# Model, loss, optimizer
# ------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SimpleSTIL(num_classes=len(CLASSES), clip_len=CLIP_LEN, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ------------------------------
# Training loop with checkpointing
# ------------------------------
best_val_acc = 0.0
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.4f} | Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    # save checkpoint
    ckpt = {
        "epoch": epoch + 1,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "val_acc": val_acc
    }
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"stil_epoch{epoch+1}.pth")
    torch.save(ckpt, ckpt_path)

    # keep best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(ckpt, os.path.join(CHECKPOINT_DIR, "best.pth"))

print("\n✅ Training complete — checkpoints saved to", CHECKPOINT_DIR)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 202MB/s]



Epoch 1/5


Train:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Val: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]


Epoch 1: Train Loss=0.7679, Train Acc=0.4000 | Val Loss=0.5763, Val Acc=1.0000

Epoch 2/5


...

Epoch 3: Train Loss=0.4331, Train Acc=0.8000 | Val Loss=0.6175, Val Acc=1.0000

Epoch 4/5


Val: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]


Epoch 4: Train Loss=0.3513, Train Acc=0.8000 | Val Loss=0.6519, Val Acc=1.0000

Epoch 5/5


Val: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s]


Epoch 5: Train Loss=0.2821, Train Acc=0.8000 | Val Loss=0.6502, Val Acc=1.0000

✅ Training complete — checkpoints saved to /content/checkpoints
